**Importing Libraries**  
This step involves importing the required Python libraries for data analysis, preprocessing, model building, training, testing, and performance evaluation. Libraries such as Pandas, NumPy, and Scikit-learn are used to implement the machine learning algorithms.

In [ ]:
!pip install -q pandas numpy scikit-learn openpyxl

**Loading the Dataset**  
In this step, the Credit Card Fraud Detection dataset obtained from Kaggle by Arif Miah is loaded into Google Colab. The dataset is read using Pandas, and the first five records, dataset dimensions, and fraud transaction distribution are displayed to understand the data.

In [ ]:
# Import pandas and numpy for data handling
import pandas as pd
import numpy as np

# Import functions for splitting and preprocessing data
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Import the three classification algorithms
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# Import evaluation metrics
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

# Display a success message
print("Libraries imported successfully!")

Libraries imported successfully!


**Data Cleaning and Preprocessing**  
Data preprocessing is performed to prepare the dataset for machine learning. Duplicate records are removed, missing target values are handled, and the target column is converted into integer format. The input features and target variable are separated for further processing.

In [ ]:
# Mount Google Drive to access the dataset
from google.colab import drive
drive.mount('/content/drive')

# Load the already-uploaded dataset
df = pd.read_csv('/content/drive/MyDrive/credit_card.csv')

# Display the number of rows and columns
print("Dataset shape:", df.shape)

# Display the first five records
display(df.head())

# Display the number of legitimate and fraudulent transactions
print(df['is_fraud'].value_counts())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset shape: (500, 10)


,transaction_id,amount,transaction_hour,merchant_category,foreign_transaction,location_mismatch,device_trust_score,velocity_last_24h,cardholder_age,is_fraud
0,1,84.47,22,Electronics,0,0,66,3,40,1
1,2,541.82,3,Travel,1,0,87,1,64,1
2,3,237.01,17,Grocery,0,0,49,1,61,1
3,4,164.33,4,Grocery,0,1,72,3,34,1
4,5,30.53,15,Food,0,0,79,0,44,1


is_fraud
0    292
1    208
Name: count, dtype: int64


**Splitting the Dataset**  
The dataset is divided into two parts: training data and testing data. 80% of the data is used for training the machine learning models, while the remaining 20% is used for testing their performance. Stratified sampling is applied to maintain the class distribution.

In [ ]:
# Remove duplicate records
df = df.drop_duplicates()

# Convert the target column to numeric values
df['is_fraud'] = pd.to_numeric(df['is_fraud'], errors='coerce')

# Remove records with a missing target value
df = df.dropna(subset=['is_fraud'])

# Convert the target into integer labels
df['is_fraud'] = df['is_fraud'].astype(int)

# Check that the target contains only 0 and 1
assert set(df['is_fraud'].unique()).issubset({0, 1}), \
    "Target must contain only 0 and 1"

# Separate input features and target
X = df.drop(columns=['is_fraud', 'transaction_id'], errors='ignore')
y = df['is_fraud']

# Display the feature and target shapes
print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Preprocessing completed!")

Features shape: (500, 8)
Target shape: (500,)
Preprocessing completed!


**Feature Preprocessing**  
In this step, numerical and categorical features are identified separately. Missing numerical values are replaced using the median, and numerical data is standardized. Categorical values are converted into numerical format using One-Hot Encoding.

In [ ]:
# Split the dataset into 80% training and 20% testing data
# Stratify preserves the class proportions as closely as possible
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Display the sizes of the four datasets
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (400, 8)
y_train: (400,)
X_test: (100, 8)
y_test: (100,)


**Building Classification Models**  
Three machine learning classification algorithms are selected for the project: Support Vector Machine (SVM), K-Nearest Neighbors (KNN), and Decision Tree. These models are created to classify credit card transactions as legitimate or fraudulent.

In [ ]:
# Identify numerical columns
num_cols = X.select_dtypes(include=np.number).columns.tolist()

# Identify categorical columns
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

# Create a function to preprocess the data
def make_preprocessor():

    # Handle missing numerical values and scale numerical features
    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    # Handle missing categorical values and encode categories
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])

    # Apply the correct preprocessing to each column type
    return ColumnTransformer([
        ('num', num_pipe, num_cols),
        ('cat', cat_pipe, cat_cols)
    ])

# Display the detected column types
print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)
print("Preprocessing pipelines created!")

Numerical columns: ['amount', 'transaction_hour', 'foreign_transaction', 'location_mismatch', 'device_trust_score', 'velocity_last_24h', 'cardholder_age']
Categorical columns: ['merchant_category']
Preprocessing pipelines created!


In [ ]:
# Install the imbalanced-learn library
!pip install -q imbalanced-learn

# Import oversampling tools
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler

# Define the three classification algorithms
models = {
    'SVM': SVC(probability=True, random_state=42),
    'KNN': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42)
}

# Create a list to store model results
results = []
trained_models = {}

# Train and evaluate each model
for name, model in models.items():

    # Create preprocessing, oversampling and classification pipeline
    clf = Pipeline([
        ('preprocessor', make_preprocessor()),
        ('oversampler', RandomOverSampler(random_state=42)),
        ('model', model)
    ])

    # Oversample only the training data and train the model
    clf.fit(X_train, y_train)

    # Predict the untouched test data
    y_pred = clf.predict(X_test)

    # Save the trained model
    trained_models[name] = clf

    # Calculate performance metrics
    accuracy = accuracy_score(y_test, y_pred) * 100
    precision = precision_score(y_test, y_pred, zero_division=0) * 100
    recall = recall_score(y_test, y_pred, zero_division=0) * 100
    f1 = f1_score(y_test, y_pred, zero_division=0) * 100

    # Store the results
    results.append({
        'Model': name,
        'Accuracy (%)': round(accuracy, 2),
        'Precision (%)': round(precision, 2),
        'Recall (%)': round(recall, 2),
        'F1 Score (%)': round(f1, 2)
    })

    # Display the results for each model
    print(f"\n--- {name} ---")
    print("Accuracy:", round(accuracy, 2), "%")
    print("Precision:", round(precision, 2), "%")
    print("Recall:", round(recall, 2), "%")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

# Display the final comparison table
comparison = pd.DataFrame(results)
display(comparison)


--- SVM ---
Accuracy: 46.0 %
Precision: 35.71 %
Recall: 35.71 %
Confusion Matrix:
[[31 27]
 [27 15]]

--- KNN ---
Accuracy: 49.0 %
Precision: 41.51 %
Recall: 52.38 %
Confusion Matrix:
[[27 31]
 [20 22]]

--- Decision Tree ---
Accuracy: 52.0 %
Precision: 41.18 %
Recall: 33.33 %
Confusion Matrix:
[[38 20]
 [28 14]]


,Model,Accuracy (%),Precision (%),Recall (%),F1 Score (%)
0,SVM,46.0,35.71,35.71,35.71
1,KNN,49.0,41.51,52.38,46.32
2,Decision Tree,52.0,41.18,33.33,36.84


### Adjusting the Classification Threshold
We will now use `predict_proba()` to retrieve raw probability scores and apply a lower custom threshold (e.g., 10%) to classify a transaction as fraud.

In [ ]:
# Set a custom threshold (e.g., 10% probability of fraud is enough to flag it)
CUSTOM_THRESHOLD = 0.10

threshold_results = []

for name, clf in trained_models.items():
    # Get probability estimates for the positive class (fraud)
    # predict_proba returns [prob_class_0, prob_class_1]
    y_probs = clf.predict_proba(X_test)[:, 1]

    # Apply custom threshold
    y_pred_custom = (y_probs >= CUSTOM_THRESHOLD).astype(int)

    # Recalculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred_custom) * 100
    precision = precision_score(y_test, y_pred_custom, zero_division=0) * 100
    recall = recall_score(y_test, y_pred_custom, zero_division=0) * 100
    f1 = f1_score(y_test, y_pred_custom, zero_division=0) * 100

    threshold_results.append({
        'Model': name,
        'Threshold': CUSTOM_THRESHOLD,
        'Accuracy (%)': round(accuracy, 2),
        'Precision (%)': round(precision, 2),
        'Recall (%)': round(recall, 2),
        'F1 Score (%)': round(f1, 2)
    })

    print(f"\n--- {name} (Threshold: {CUSTOM_THRESHOLD}) ---")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred_custom, labels=[0, 1]))

# Display the adjusted performance results
threshold_comparison = pd.DataFrame(threshold_results)
display(threshold_comparison)


--- SVM (Threshold: 0.1) ---
Confusion Matrix:
[[ 0 58]
 [ 0 42]]

--- KNN (Threshold: 0.1) ---
Confusion Matrix:
[[ 6 52]
 [ 6 36]]

--- Decision Tree (Threshold: 0.1) ---
Confusion Matrix:
[[38 20]
 [28 14]]


,Model,Threshold,Accuracy (%),Precision (%),Recall (%),F1 Score (%)
0,SVM,0.1,42.0,42.00,100.00,59.15
1,KNN,0.1,42.0,40.91,85.71,55.38
2,Decision Tree,0.1,52.0,41.18,33.33,36.84


**Training and Testing the Models Using Oversampling**  
The three classification models are trained using the training dataset. Random Oversampling is applied to increase the representation of fraudulent transactions in the training data. Each trained model is then tested using the separate testing dataset to predict fraudulent and legitimate transactions.

In [ ]:
# Create empty lists and dictionaries to store results
results = []
trained_models = {}

# Train and evaluate each model
for name, model in models.items():

    # Combine preprocessing and classification into one pipeline
    clf = Pipeline([
        ('preprocessor', make_preprocessor()),
        ('model', model)
    ])

    # Train the model using training data
    clf.fit(X_train, y_train)

    # Predict transaction classes for the test data
    y_pred = clf.predict(X_test)

    # Save the trained pipeline
    trained_models[name] = clf

    # Calculate accuracy, precision, recall and F1-score
    accuracy = accuracy_score(y_test, y_pred) * 100
    precision = precision_score(y_test, y_pred, zero_division=0) * 100
    recall = recall_score(y_test, y_pred, zero_division=0) * 100
    f1 = f1_score(y_test, y_pred, zero_division=0) * 100

    # Store the evaluation metrics
    results.append({
        'Model': name,
        'Accuracy (%)': round(accuracy, 2),
        'Precision (%)': round(precision, 2),
        'Recall (%)': round(recall, 2),
        'F1 Score (%)': round(f1, 2)
    })

    # Display the evaluation results for the current model
    print(f"\n--- {name} ---")
    print("Accuracy:", round(accuracy, 2), "%")
    print("Precision:", round(precision, 2), "%")
    print("Recall:", round(recall, 2), "%")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

# Confirm that all three models have been evaluated
print("\nAll three models trained and tested!")


--- SVM ---
Accuracy: 54.0 %
Precision: 30.0 %
Recall: 7.14 %
Confusion Matrix:
[[51  7]
 [39  3]]

--- KNN ---
Accuracy: 51.0 %
Precision: 40.54 %
Recall: 35.71 %
Confusion Matrix:
[[36 22]
 [27 15]]

--- Decision Tree ---
Accuracy: 44.0 %
Precision: 36.0 %
Recall: 42.86 %
Confusion Matrix:
[[26 32]
 [24 18]]

All three models trained and tested!


**Performance Evaluation and Model Comparison**  
The performance of all three models is evaluated using accuracy, precision, recall, and F1-score. A confusion matrix is also generated to understand the correct and incorrect predictions. The results are displayed in a comparison table to analyze the performance of the models.

In [ ]:
# Convert the stored results into a DataFrame
comparison = pd.DataFrame(results)

# Display the required comparison table
display(
    comparison[
        ['Model', 'Accuracy (%)', 'Precision (%)', 'Recall (%)']
    ]
)

,Model,Accuracy (%),Precision (%),Recall (%)
0,SVM,54.0,30.00,7.14
1,KNN,51.0,40.54,35.71
2,Decision Tree,44.0,36.00,42.86


**Saving and Downloading Results**  
The performance comparison results of SVM, KNN, and Decision Tree are saved in an Excel file. This file can be downloaded and used in the project report for documentation and analysis.

In [ ]:
# Save the comparison results as an Excel file
comparison.to_excel('model_comparison.xlsx', index=False)

# Import the Google Colab download utility
from google.colab import files

# Download the Excel comparison table to your computer
files.download('model_comparison.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Conclusion**  
In this final step, the results of all three classification algorithms are displayed and compared. The model performance is analyzed using accuracy, precision, and recall to understand how effectively each algorithm identifies fraudulent credit card transactions.

In [ ]:
# Print the project title
print("PROJECT: Credit Card Fraud Detection Using Machine Learning")

# Display the comparison of all three models
print("\nModel Performance Comparison:")
print(comparison.to_string(index=False))

# Explain the evaluation approach
print("\nConclusion:")
print("SVM, KNN and Decision Tree were trained and tested.")
print("Compare precision, recall and confusion matrices")
print("to understand how well each model detects fraud.")

# Remind the user about the dataset limitation
print("\nNote: Very few fraud examples can make the results unstable.")

PROJECT: Credit Card Fraud Detection Using Machine Learning

Model Performance Comparison:
        Model  Accuracy (%)  Precision (%)  Recall (%)  F1 Score (%)
          SVM          54.0          30.00        7.14         11.54
          KNN          51.0          40.54       35.71         37.97
Decision Tree          44.0          36.00       42.86         39.13

Conclusion:
SVM, KNN and Decision Tree were trained and tested.
Compare precision, recall and confusion matrices
to understand how well each model detects fraud.

Note: Very few fraud examples can make the results unstable.
